In [0]:

%sql
CREATE OR REPLACE TABLE workspace.finance.merchant_rules (
  pattern       STRING  COMMENT 'Text to look for in the description (case-insensitive)',
  merchant_name STRING  COMMENT 'Clean display name',
  category      STRING,
  priority      INT     COMMENT 'Lower number wins when several rules match'
);

In [0]:
%sql
INSERT INTO workspace.finance.merchant_rules VALUES
  ('SALARY',              'Salary',              'Salary',        1),
  ('REFUND',              'Refund',              'Refund',        1),
  ('CREDIT CARD PAYMENT', 'Credit Card Payment', 'Transfer',      1),
  ('PAYMENT RECEIVED',    'Credit Card Payment', 'Transfer',      1),
  ('HOUSE RENT',          'Rent',                'Rent',          5),
  ('SWIGGY',              'Swiggy',              'Food & Dining', 10),
  ('ZOMATO',              'Zomato',              'Food & Dining', 10),
  ('STARBUCKS',           'Starbucks',           'Food & Dining', 10),
  ('BARBEQUE NATION',     'Barbeque Nation',     'Food & Dining', 10),
  ('BIGBASKET',           'BigBasket',           'Groceries',     10),
  ('BLINKIT',             'Blinkit',             'Groceries',     10),
  ('UBER',                'Uber',                'Transport',     10),
  ('OLA CABS',            'Ola',                 'Transport',     10),
  ('PETROL',              'Fuel',                'Transport',     20),
  ('AMAZON',              'Amazon',              'Shopping',      10),
  ('FLIPKART',            'Flipkart',            'Shopping',      10),
  ('MYNTRA',              'Myntra',              'Shopping',      10),
  ('CROMA',               'Croma',               'Shopping',      10),
  ('NETFLIX',             'Netflix',             'Entertainment', 10),
  ('SPOTIFY',             'Spotify',             'Entertainment', 10),
  ('BOOKMYSHOW',          'BookMyShow',          'Entertainment', 10),
  ('AIRTEL',              'Airtel',              'Utilities',     10),
  ('ELECTRICITY',         'Electricity Board',   'Utilities',     10),
  ('APOLLO PHARMACY',     'Apollo Pharmacy',     'Health',        10),
  ('CULT FIT',            'Cult.fit',            'Health',        10),
  ('MAKEMYTRIP',          'MakeMyTrip',          'Travel',        10);

num_affected_rows,num_inserted_rows
26,26


In [0]:
%sql
CREATE OR REPLACE TABLE workspace.finance.transactions_categorized AS
WITH matches AS (
  SELECT
    t.*,
    r.pattern, r.merchant_name, r.category AS rule_category,
    ROW_NUMBER() OVER (
      PARTITION BY t.txn_id
      ORDER BY r.priority ASC NULLS LAST, length(r.pattern) DESC NULLS LAST
    ) AS rn
  FROM workspace.finance.silver_transactions t
  LEFT JOIN workspace.finance.merchant_rules r
    ON upper(t.description) LIKE concat('%', upper(r.pattern), '%')
),
best AS (SELECT * FROM matches WHERE rn = 1)
SELECT
  txn_id, account, txn_date, description, amount, direction, source_file,
  -- readable merchant: rule name if matched, otherwise cleaned from the description
  coalesce(merchant_name, initcap(trim(regexp_replace(
      CASE WHEN upper(description) LIKE 'UPI/%' THEN split(description, '/')[1]
           ELSE regexp_replace(regexp_replace(upper(description),
                  '^(POS|NEFT|IMPS|ACH|BILLDESK)[ /-]+', ''), '[^A-Z ]', ' ')
      END, '[ ]+', ' ')))) AS merchant,
  CASE WHEN rule_category IS NOT NULL THEN rule_category
       WHEN direction = 'CREDIT'      THEN 'Other Income'
       ELSE 'Uncategorized' END AS category,
  CASE WHEN rule_category IS NOT NULL THEN 'rule' ELSE 'default' END AS category_source,
  pattern AS matched_rule,
  date_trunc('MONTH', txn_date) AS txn_month
FROM best;

num_affected_rows,num_inserted_rows


In [0]:
%sql
SELECT count(*) FROM workspace.finance.transactions_categorized;   -- must still be 272

SELECT category, count(*) AS txns, sum(amount) AS total
FROM workspace.finance.transactions_categorized
GROUP BY category ORDER BY total DESC;

category,txns,total
Salary,6,570000.00
Transfer,10,336688.92
Shopping,31,127898.70
Rent,6,108000.00
Food & Dining,90,51908.59
Groceries,31,23442.68
Health,18,22414.74
Transport,37,20306.21
Utilities,12,15427.82
Travel,1,12480.00


In [0]:
%sql
SELECT
  sum(CASE WHEN direction = 'DEBIT' THEN amount END)                            AS naive_spending,
  sum(CASE WHEN direction = 'DEBIT' AND category <> 'Transfer' THEN amount END) AS real_spending
FROM workspace.finance.transactions_categorized;

naive_spending,real_spending
562148.23,393803.77


In [0]:
%sql
SELECT merchant, count(*) AS txns, sum(amount) AS total, first(description) AS example
FROM workspace.finance.transactions_categorized
WHERE category = 'Uncategorized'
GROUP BY merchant;

merchant,txns,total,example
Sri Lakshmi Tea Stall,3,515.21,UPI/SRI LAKSHMI TEA STALL/paytmqr@paytm/Payment/270541592
Green Leaf Vegetables,3,798.66,UPI/GREEN LEAF VEGETABLES/paytmqr@paytm/Payment/128063847
Ravi Stores,3,496.52,UPI/RAVI STORES/paytmqr@paytm/Payment/175666412
The Book Nook,5,4011.59,POS 453212******1234 THE BOOK NOOK


In [0]:
%sql
CREATE OR REPLACE TABLE workspace.finance.ai_merchant_categories AS
SELECT
  merchant,
  ai_classify(
    concat('Merchant: ', merchant, '. Bank description: ', example),
    ARRAY('Food & Dining', 'Groceries', 'Transport', 'Shopping', 'Entertainment',
          'Utilities', 'Health', 'Travel', 'Rent', 'Other')
  ) AS ai_category
FROM (
  SELECT merchant, first(description) AS example
  FROM workspace.finance.transactions_categorized
  WHERE direction = 'DEBIT' AND category <> 'Transfer'
  GROUP BY merchant
);

SELECT * FROM workspace.finance.ai_merchant_categories;

merchant,ai_category
Starbucks,Food & Dining
Amazon,Shopping
Myntra,Shopping
Airtel,Utilities
Swiggy,Food & Dining
Fuel,Transport
Flipkart,Shopping
Barbeque Nation,Food & Dining
Electricity Board,Utilities
Apollo Pharmacy,Health


In [0]:
%sql
SELECT
  count(*)                                               AS merchants_with_rules,
  count_if(r.category = a.ai_category)                   AS ai_agrees,
  round(100.0 * count_if(r.category = a.ai_category) / count(*), 1) AS agreement_pct
FROM (SELECT DISTINCT merchant, category FROM workspace.finance.transactions_categorized
      WHERE category_source = 'rule' AND direction = 'DEBIT' AND category <> 'Transfer') r
JOIN workspace.finance.ai_merchant_categories a USING (merchant);

merchants_with_rules,ai_agrees,agreement_pct
22,22,100.0


In [0]:
%sql
MERGE INTO workspace.finance.transactions_categorized AS t
USING workspace.finance.ai_merchant_categories AS a
ON t.merchant = a.merchant AND t.category = 'Uncategorized'
WHEN MATCHED THEN UPDATE SET
  t.category = a.ai_category,
  t.category_source = 'ai';

SELECT category_source, count(*) FROM workspace.finance.transactions_categorized GROUP BY 1;

category_source,count(*)
rule,258
ai,14
